## Install required package

In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.6 MB/s eta 0:00:00


## Importing libraries

In [2]:
import os
import glob
import json
import shutil
import cv2
from tqdm import tqdm
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## Initial Configuration

In [3]:
WORK_DIR = "/kaggle/working/datasets/remote_sensing"

for split in ['train', 'val']:
    os.makedirs(f"{WORK_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{WORK_DIR}/labels/{split}", exist_ok=True)

## Data parsing Engine

In [4]:
os.chdir('/kaggle/input/datasets/chandlertimm/dota-data/DOTA')
os.listdir()

['val', 'test', 'train']

In [5]:
CLASS_MAP = {
    'plane': 0, 'ship': 1, 'storage-tank': 2, 'baseball-diamond': 3,
    'tennis-court': 4, 'basketball-court': 5, 'ground-track-field': 6,
    'harbor': 7, 'bridge': 8, 'large-vehicle': 9, 'small-vehicle': 10,
    'helicopter': 11, 'roundabout': 12, 'soccer-ball-field': 13, 
    'swimming-pool': 14, 'container-crane': 15, 'airport': 16, 'helipad': 17,
    'building': 18, 'road': 19, 'water': 20
}

def get_image_dims(img_path):
    img = cv2.imread(img_path)
    return (img.shape[1], img.shape[0]) if img is not None else (1024, 1024)

# 2. Native Kaggle DOTA Parser
# 2. Native Kaggle DOTA Parser
def process_kaggle_dota(split):
    print(f"\nProcessing DOTA {split}...")
    
    # Explicitly define base paths using your exact mount locations
    img_dir = f"/kaggle/input/datasets/chandlertimm/dota-data/DOTA/{split}/images"
    lbl_base = "/kaggle/input/datasets/dlfelps/dota-labels/dota_labels/obb"
    
    # Search for the split folder (train/val) specifically within the 'obb' directory
    lbl_searches = [d for d in glob.glob(f"{lbl_base}/**/*{split}*", recursive=True) if os.path.isdir(d)]
    
    if not os.path.exists(img_dir) or not lbl_searches:
        print(f"Skipping DOTA {split}: Could not find mounted image or label directories.")
        return
        
    lbl_dir = lbl_searches[0]
    
    txt_files = glob.glob(f"{lbl_dir}/**/*.txt", recursive=True)
    lbl_map = {os.path.splitext(os.path.basename(p))[0]: p for p in txt_files}
    
    print(f"Found {len(lbl_map)} DOTA label files. Normalizing boxes...")
    
    matched_images = 0
    total_objects = 0

    for img_path in tqdm(glob.glob(f"{img_dir}/*.png")):
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        
        if base_name in lbl_map:
            target_img = os.path.join(WORK_DIR, "images", split, os.path.basename(img_path))
            target_lbl = os.path.join(WORK_DIR, "labels", split, f"{base_name}.txt")
            
            # Copy image
            if not os.path.exists(target_img):
                shutil.copy(img_path, target_img)
                
            # Parse & Write Label
            img_w, img_h = get_image_dims(img_path)
            yolo_lines = []
            
            with open(lbl_map[base_name], 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    
                    # Ensure it's a valid DOTA 8-point polygon line
                    if len(parts) >= 9:
                        cls_name = parts[8].lower()
                        if cls_name in CLASS_MAP:
                            try:
                                xs = [float(x) for x in parts[0:8:2]]
                                ys = [float(y) for y in parts[1:8:2]]
                                xmin, xmax = min(xs), max(xs)
                                ymin, ymax = min(ys), max(ys)
                                
                                x_center = max(0.0, min(1.0, ((xmin + xmax) / 2) / img_w))
                                y_center = max(0.0, min(1.0, ((ymin + ymax) / 2) / img_h))
                                width = max(0.0, min(1.0, (xmax - xmin) / img_w))
                                height = max(0.0, min(1.0, (ymax - ymin) / img_h))
                                
                                if width > 0 and height > 0:
                                    cls_id = CLASS_MAP[cls_name]
                                    yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
                                    total_objects += 1
                            except ValueError:
                                continue # Skip corrupted coordinate lines
                                
            with open(target_lbl, 'w') as f:
                f.writelines(yolo_lines)
            matched_images += 1

    print(f"Processed {matched_images} DOTA images | Created {total_objects} bounding boxes.")
    
# 3. VRSBench Parser
def process_vrsbench(split, json_file):
    print(f"\nProcessing VRSBench {split}...")
    vrs_search = glob.glob(f"/kaggle/input/**/vrsbench_data", recursive=True)
    
    if not vrs_search:
        print(f"Skipping VRSBench {split}: Folder not found.")
        return
        
    vrs_base = vrs_search[0]
    img_dir = os.path.join(vrs_base, f"Images_{split}")
    json_path = os.path.join(vrs_base, json_file)

    with open(json_path, 'r') as f:
        vrs_data = json.load(f)

    vrs_boxes = 0
    for item in tqdm(vrs_data):
        img_name = item.get("image_name") or item.get("image_id")
        img_path = os.path.join(img_dir, str(img_name))

        if os.path.exists(img_path):
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            target_img = os.path.join(WORK_DIR, "images", split, os.path.basename(img_path))
            target_lbl = os.path.join(WORK_DIR, "labels", split, f"{base_name}.txt")

            if not os.path.exists(target_img):
                shutil.copy(img_path, target_img)

            img_w, img_h = get_image_dims(img_path)
            bbox = item.get("bbox", [])
            cls_name = item.get("class_name", "building").lower()

            if len(bbox) == 4 and cls_name in CLASS_MAP:
                ymin, xmin, ymax, xmax = bbox
                
                x_center = max(0.0, min(1.0, ((xmin + xmax) / 2) / img_w))
                y_center = max(0.0, min(1.0, ((ymin + ymax) / 2) / img_h))
                width = max(0.0, min(1.0, (xmax - xmin) / img_w))
                height = max(0.0, min(1.0, (ymax - ymin) / img_h))

                if width > 0 and height > 0:
                    cls_id = CLASS_MAP[cls_name]
                    with open(target_lbl, 'a') as f:
                        f.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
                    vrs_boxes += 1

    print(f"Added {vrs_boxes} VRSBench boxes.")

# Execute pipelines
process_kaggle_dota('train')
process_kaggle_dota('val')
process_vrsbench('train', 'VRSBench_train.json')
process_vrsbench('val', 'VRSBench_EVAL_referring.json')

print("Data Extraction Complete!")


Processing DOTA train...
Found 1830 DOTA label files. Normalizing boxes...


100%|██████████| 1411/1411 [07:04<00:00,  3.32it/s]


Processed 1411 DOTA images | Created 210631 bounding boxes.

Processing DOTA val...
Found 593 DOTA label files. Normalizing boxes...


100%|██████████| 458/458 [02:11<00:00,  3.48it/s]


Processed 458 DOTA images | Created 69565 bounding boxes.

Processing VRSBench train...


100%|██████████| 142390/142390 [00:00<00:00, 213351.27it/s]


Added 0 VRSBench boxes.

Processing VRSBench val...


100%|██████████| 16159/16159 [00:09<00:00, 1620.81it/s]

Added 0 VRSBench boxes.
Data Extraction Complete!


## Generating the YOLO YAML Config

In [6]:
yaml_path = os.path.join(WORK_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(f"path: {WORK_DIR}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n\n")
    f.write(f"nc: {len(CLASS_MAP)}\n")
    f.write(f"names: {list(CLASS_MAP.keys())}\n")

print(f"\nConfiguration written to {yaml_path}")


Configuration written to /kaggle/working/datasets/remote_sensing/data.yaml


## Train YOLO11

In [7]:
os.chdir('/kaggle/working')

yaml_path = "/kaggle/working/datasets/remote_sensing/data.yaml"
print(f"Initializing YOLO with config: {yaml_path}")

model = YOLO("yolo11s.pt")

# Train the model
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=800,
    batch=16,
    device=0,
    workers=4,
    cache='ram',
    project='/kaggle/working/runs',
    name='satquery_region_highlighting'
)

Initializing YOLO with config: /kaggle/working/datasets/remote_sensing/data.yaml
Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/datasets/remote_sensing/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.p

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      14.3G       1.91      1.387     0.9986        465        800: 100% ━━━━━━━━━━━━ 89/89 2.1it/s 42.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.1s/it 31.0s
                   all        458      69565      0.633      0.272      0.267      0.153

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      1.44G      1.853       1.24     0.9745        302        800: 100% ━━━━━━━━━━━━ 89/89 2.1it/s 41.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.7s/it 26.0s
                   all        458      69565      0.631      0.287      0.308      0.174

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      13.3G      1.771      1.145     0.9557        122        800: 100% ━━━━━━━━━━━━ 89/89 2.2it/s 40.6s
                 Class     Images  Instances      Box(P  

## Metrics Calculation

In [8]:
best_weights = "/kaggle/working/runs/satquery_region_highlighting/weights/best.pt"
yaml_path = "/kaggle/working/datasets/remote_sensing/data.yaml"

print("Loading best model weights...")
val_model = YOLO(best_weights)

# Run validation sequence
print("\n--- RUNNING FINAL VALIDATION BENCHMARK ---")
metrics = val_model.val(data=yaml_path, imgsz=800, batch=16, device=0)

# Output core metrics
print("\nVALIDATION METRICS:")
print(f"mAP@50 (Mean Average Precision): {metrics.box.map50:.4f}")
print(f"mAP@50-95 (Strict Precision):    {metrics.box.map:.4f}")
print(f"Precision (Confidence):          {metrics.box.mp:.4f}")
print(f"Recall (Detection Rate):         {metrics.box.mr:.4f}")

Loading best model weights...

--- RUNNING FINAL VALIDATION BENCHMARK ---
Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,420,927 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1909.5±1394.8 MB/s, size: 16746.5 KB)
val: Scanning /kaggle/working/datasets/remote_sensing/labels/val.cache... 458 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 458/458 192.1Mit/s 0.0s
WARNING ⚠️ Dataset images contain up to 8609 objects (val=8609), but max_det=300. This mismatch can cap recall and produce invalid validation metrics. Raising it may increase validation cost but cannot increase model or export capacity, which may cap recall. Setting max_det=8609 to match the observed maximum.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.0s/it 57.0s
                   all        458      69565      0.768      0.453      0.481

In [9]:
!pip install -q huggingface_hub

In [10]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# Authenticate via Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
except Exception as e:
    print("Authentication failed. Add 'HF_TOKEN' to Kaggle Secrets.")
    raise e

repo_id = "nillohitroy/satquery-yolo11-region-highlighting"
api = HfApi()

print(f"Creating repository: {repo_id}...")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

print("Uploading best.pt weights...")
api.upload_file(
    path_or_fileobj="/kaggle/working/runs/satquery_region_highlighting/weights/best.pt",
    path_in_repo="best.pt",
    repo_id=repo_id,
    repo_type="model"
)

print("Uploading data.yaml taxonomy...")
api.upload_file(
    path_or_fileobj="/kaggle/working/datasets/remote_sensing/data.yaml",
    path_in_repo="data.yaml",
    repo_id=repo_id,
    repo_type="model"
)

print(f"Upload complete! Weights are live at: https://huggingface.co/{repo_id}")

Creating repository: nillohitroy/satquery-yolo11-region-highlighting...
Uploading best.pt weights...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading data.yaml taxonomy...
Upload complete! Weights are live at: https://huggingface.co/nillohitroy/satquery-yolo11-region-highlighting
